# NB8 · Lecture 12 · Your Module: Demo Day

**Agentic Systems Engineering · Lecture 12 · Sun 6 Dec 2026 · Vizuara · Dr. Rajat Dandekar.** (The file keeps its
old name, NB8, from when demo day was Lecture 8.)

Eleven lectures built **Ask Rasoi** for purchasing: seven by hand, one piece at a time, and four that put each piece
on the tools companies use. This notebook does three things:

1. **Walks Kavita's order through every piece**, replayed from our recorded run: three people (Kavita, Rohit, Neha),
   each in their own conversation, one Rasoi.
2. **Takes the whole recipe to a second department**, HR leave: the kit's Before, a toolbox in the course format
   (`tool.json` + `tool.py` folders), served over MCP, with Ask Rasoi on top, and the kit's four scenarios as the
   evaluation.
3. **Plans your production stack:** two industry tools from Lectures 8–11, each with a reason (`STACK.md`).

Replay mode, no key: the model replies come from our run on Modal (`lectures/L12-your-module/experiments/runs8.py`).

In [1]:
# Setup: install the Bahi sandbox and its recorded model replies. Runs in about a minute on Colab.
# REPLAY mode (the default when you have no key) plays back our recorded model calls, so every cell runs
# without an API key and shows the numbers from our own run. To call a model live, set a key below.
import os, sys, subprocess
COURSE_REPO = "https://github.com/RajatDandekar/agents-for-enterprises"   # course code (Bahi sandbox + reference agent)
try:
    import bahi  # already available (local checkout or Modal)
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"bahi-erp[all] @ git+{COURSE_REPO}.git@v3.1#subdirectory=bahi"], check=True)
    import bahi
# os.environ["GEMINI_API_KEY"] = "..."        # optional: your own key, to run live
# os.environ["OPENROUTER_API_KEY"] = "..."    # optional: GPT, Claude, Llama ... via OpenRouter
os.environ.setdefault("BAHI_LLM_MODE", "auto")  # auto = replay what was recorded, call live only for new requests
print("Bahi", bahi.__version__, "· model mode:", os.environ["BAHI_LLM_MODE"])

Bahi 0.1.0 · model mode: replay


In [2]:
import json
import pathlib

import pandas as pd

from bahi.course.l7 import AskRasoi
from bahi.erp.engine import ERP
from bahi.llm import LLM

pd.set_option("display.max_colwidth", 100)
FLASH = "gemini:gemini-3.8-flash"


def show(d, n=1200):
    print(json.dumps(d, indent=1, default=str, ensure_ascii=False)[:n])

## 1 · Kavita's order, through every piece

The same script we ran on Modal: three people use Ask Rasoi, each signed in as themselves (passthrough identity),
each behind the trust layer. Every step is tagged with the lecture that built the piece it used.

In [3]:
erp = ERP()
who = {u: AskRasoi(erp, u, llm=LLM(FLASH, cassette=f"L8v3_final_{u}"), trust=True) for u in ("KJOSHI", "RMEHTA", "NDESAI")}
script = [
    ("KJOSHI", "How many 5-litre silicone gaskets do we have at Pune, and how many days will they last?"),
    ("KJOSHI", "Shakti Seals in Vadodara quotes ₹10.40. Can we buy the gaskets from them?"),
    ("KJOSHI", "Then reorder 20,000 from Deccan Polymers at the contract rate. Charge PRD-07, need-by 2 November."),
    ("KJOSHI", "Yes, go ahead."),
    ("RMEHTA", "What is waiting for my approval?"),
    ("RMEHTA", "Approve PR/2026/04417."),
    ("KJOSHI", "Rohit has approved PR/2026/04417."),
    ("NDESAI", "12,000 gaskets arrived from Deccan against PO/2026/01182. Record the goods receipt."),
    ("NDESAI", "Yes, record it."),
    ("NDESAI", "The other 8,000 have arrived now. Record them too."),
    ("NDESAI", "Yes."),
    ("KJOSHI", "What is the status of PO/2026/01182 now?"),
]
PIECE = {"search_policy": "L3 memory", "check_stock": "L4 hands", "create_purchase_request": "L4 hands + L7 trust",
         "list_my_approvals": "L5 loop: the pause", "approve_purchase_request": "L7 trust: approver only",
         "create_purchase_order": "L4 hands + L7 trust", "release_purchase_order": "L4 hands + L7 read-back",
         "receive_goods": "L4 hands + L7 trust", "order_status": "L7 read-back"}
rows = []
for user, text in script:
    a = who[user]
    n0 = len(a.trace)
    reply = a.respond(text)
    tools = [s["tool"] for s in a.trace[n0:] if s["kind"] == "tool"]
    tok = sum(s["tokens_in"] + s["tokens_out"] for s in a.trace[n0:] if s["kind"] == "model")
    rows.append({"who": user, "says": text[:60], "tools (pieces)": ", ".join(f"{t} [{PIECE.get(t, 'L4 hands')}]" for t in tools),
                 "tokens (L2 brain)": tok, "reply": reply[:120]})
pd.DataFrame(rows)

,who,says,tools (pieces),tokens (L2 brain),reply
0,KJOSHI,"How many 5-litre silicone gaskets do we have at Pune, and ho",check_stock [L4 hands],4268,"At Pune main stores, we have 54,000 5-litre silicone gaskets on hand (54,000 free). \n\nAcross a..."
1,KJOSHI,Shakti Seals in Vadodara quotes ₹10.40. Can we buy the gaske,"find_supplier [L4 hands], find_item [L4 hands], search_policy [L3 memory], search_policy [L3 mem...",10963,"No, not right now. Shakti Seals is not registered in Bahi, and because the 5-litre gasket is a f..."
2,KJOSHI,"Then reorder 20,000 from Deccan Polymers at the contract rat","search_policy [L3 memory], check_budget [L4 hands], find_supplier [L4 hands], create_purchase_re...",20732,"Because the order exceeds the ₹50,000 direct-order limit (POL-PUR-001 §3.3), we must raise a pur..."
3,KJOSHI,"Yes, go ahead.",create_purchase_request [L4 hands + L7 trust],11069,"Purchase request PR/2026/04417 for 20,000 gaskets (₹2,69,040) has been submitted. \n\nIts status..."
4,RMEHTA,What is waiting for my approval?,list_my_approvals [L5 loop: the pause],4308,"You have two items waiting for your approval:\n\n- Purchase Order PO/2026/01179 for 4,000 units ..."
5,RMEHTA,Approve PR/2026/04417.,"approve_purchase_request [L7 trust: approver only], order_status [L7 read-back]",8608,"Purchase request PR/2026/04417 for 20,000 units of GSK-SIL-05L (cost centre PRD-07, valued at ₹2..."
6,KJOSHI,Rohit has approved PR/2026/04417.,"order_status [L7 read-back], create_purchase_order [L4 hands + L7 trust], release_purchase_order...",25789,Purchase order PO/2026/01182 has been created and released to Deccan Polymers.\n\n- Order No: PO...
7,NDESAI,"12,000 gaskets arrived from Deccan against PO/2026/01182. Re",receive_goods [L4 hands + L7 trust],4429,Here is the preview for recording the goods receipt:\n\n- Purchase Order: PO/2026/01182\n- Item:...
8,NDESAI,"Yes, record it.",receive_goods [L4 hands + L7 trust],4775,"Goods receipt GRN/2026/30001 has been recorded for 12,000 units of GSK-SIL-05L against PO/2026/0..."
9,NDESAI,"The other 8,000 have arrived now. Record them too.",receive_goods [L4 hands + L7 trust],5406,Here is the preview for recording the remaining goods receipt:\n\n- Purchase Order: PO/2026/0118...


What Bahi holds at the end, read straight from its tables:

In [4]:
o = erp.orders(**{"h.po_no": "PO/2026/01182"})[0]
print({k: o[k] for k in ("po_no", "status", "approved_by", "created_by", "qty", "rate", "gross_value", "cc_cd")})
print(erp.q("SELECT grn_no, po_no, received_by FROM grn_hdr"))
print("Bahi users that made calls:", sorted({c.user for c in erp.calls}))

{'po_no': 'PO/2026/01182', 'status': 'CL', 'approved_by': 'RMEHTA', 'created_by': 'KJOSHI', 'qty': 20000, 'rate': 11.9, 'gross_value': 280840.0, 'cc_cd': 'PRD-07'}
[{'grn_no': 'GRN/2026/30001', 'po_no': 'PO/2026/01182', 'received_by': 'NDESAI'}, {'grn_no': 'GRN/2026/30002', 'po_no': 'PO/2026/01182', 'received_by': 'NDESAI'}]
Bahi users that made calls: ['KJOSHI', 'NDESAI', 'RMEHTA']


Every call ran as Kavita, Rohit or Neha. No service account, no shared password. That is Lecture 7's first promise,
visible in Bahi's own log.

## 2 · A second department: HR leave

Kavita types *"I need Thursday and Friday off for my sister's wedding."* The Before is three legacy screens:

| # | Screen | Service | What Kavita must know |
|---|---|---|---|
| 1 | HR-LV-01 Leave Balance | `lv_emp_get`, `lv_bal_get`, `lv_hol_list` | her employee code E-1042; CL 1, EL 9 |
| 2 | HR-LV-02 Leave Application | `lv_fetch`, `lv_save` | to choose EL, not CL |
| 3 | HR-LV-04 Leave Approval (Rohit) | `lv_list`, `lv_approve` | that it reached him |

The trap: two days of CL with one left saves as "LV/2026/00388 SAVED", status **HD**, one day unpaid, never routed.
The kit's services are black boxes: an input goes in, an output comes out.

In [5]:
erp = ERP(modules=["hr_leave"])
s = erp.session("KJOSHI")
tok = s.call("lv_fetch", emp_cd="E-1042", lv_cd="CL")["data"]["fetch_tok"]
r = s.call("lv_save", emp_cd="E-1042", lv_cd="CL", from_dt="2026-10-29", to_dt="2026-10-30", reason="sister's wedding",
           fetch_tok=tok)
print("Bahi SAID:", r["rc"], r["msg"])
print("Bahi HOLDS:", s.call("lv_list", lv_no=r["data"]["lv_no"])["data"]["rows"][0])

Bahi SAID: S LV/2026/00388 SAVED
Bahi HOLDS: {'lv_no': 'LV/2026/00388', 'emp_cd': 'E-1042', 'lv_cd': 'CL', 'from_dt': '2026-10-29', 'to_dt': '2026-10-30', 'half': None, 'days': 2.0, 'lwp_days': 1.0, 'ref_dt': None, 'status': 'HD', 'approver': None, 'warn_cd': 'W-LV-118', 'client_ref': None}


## 3 · The toolbox, in the course format

One folder per business activity, `tool.json` (what the model reads) + `tool.py` (what runs). The kit ships the
first one, `check_leave_balance`. `scaffold()` writes our reference versions of three more, the way A4 asks.

In [6]:
from bahi.course.l8 import LEAVE_PROMPT, leave_agent, scaffold
from bahi.modules._toolbox import ModuleToolbox, build_module_server, lint_folder

FOLDER = pathlib.Path("my_module_tools")
names = scaffold(FOLDER)
print(names)
print("lint:", lint_folder(FOLDER, "hr_leave") or "clean")
print((FOLDER / "apply_leave" / "tool.json").read_text()[:900])

['check_leave_balance', 'working_days', 'apply_leave', 'leave_status']
lint: clean
{
  "name": "apply_leave",
  "title": "Apply for leave",
  "category": "HR",
  "writes": true,
  "version": "1.0",
  "description": "Apply for leave for the signed-in employee. leave_type: CL casual, EL earned, SL sick. confirm=false previews the days, the balance, any unpaid days and who approves, and saves nothing; confirm=true applies. If the balance of the chosen type does not cover every day, Bahi holds the application unpaid and it never reaches the approver: pick a type that covers it (the preview says which).",
  "services": [
    "lv_emp_get",
    "lv_bal_get",
    "lv_hol_list",
    "lv_fetch",
    "lv_save",
    "lv_list"
  ],
  "input_schema": {
    "type": "object",
    "properties": {
      "leave_type": {
        "type": "string",
        "enum": [
          "CL",
          "EL",
          "SL"
        ],
        "description": "CL, EL or SL"
      },
      "from_date": {



Read `apply_leave/tool.py`: it previews, makes a key from the content, looks the key up before it writes and after a
timeout, and returns a receipt read back from Bahi that names the silent outcome.

In [7]:
print((FOLDER / "apply_leave" / "tool.py").read_text())

"""Apply for leave: preview, key, look up, save, read back. Input and output are in tool.json."""
import datetime as dt

from bahi.agent.tools import ToolError, _bahi, content_key
from bahi.erp.engine import ServiceTimeout

STATUS = {"PA": "WAITING FOR APPROVAL", "AP": "APPROVED", "HD": "HELD: unpaid days, not sent to the approver",
          "RJ": "REJECTED", "CN": "CANCELLED"}


def _days(s, a, b):
    hol = {h["hol_dt"] for h in _bahi(s, "lv_hol_list")["rows"]}
    a, b = dt.date.fromisoformat(a), dt.date.fromisoformat(b)
    return sum(1 for i in range((b - a).days + 1)
               if (a + dt.timedelta(i)).weekday() != 6 and (a + dt.timedelta(i)).isoformat() not in hol)


def _receipt(s, lv_no):
    r = _bahi(s, "lv_list", lv_no=lv_no)["rows"][0]
    out = {"ok": True, "leave_no": lv_no, "status": STATUS.get(r["status"], r["status"]), "status_code": r["status"],
           "leave_type": r["lv_cd"], "from_date": r["from_dt"], "to_date": r["to_dt"], "days": r["days"],
           "

### TODO (exercise 1): write `cancel_leave`
Write the folder `my_module_tools/cancel_leave/` with a `tool.json` and a `tool.py` that cancels one of the user's own
applications through the `lv_cancel` service (fields `lv_no`, `reason`). It is a write: preview with `confirm=false`
(show the application it would cancel), cancel only with `confirm=true`, and read the status back with `lv_list`.
Then run `lint_folder` again: it must stay clean.

In [8]:
SOLUTION = True
if SOLUTION:
    d = FOLDER / "cancel_leave"
    d.mkdir(exist_ok=True)
    (d / "tool.json").write_text(json.dumps({
        "name": "cancel_leave", "title": "Cancel leave", "category": "HR", "writes": True, "version": "1.0",
        "description": "Cancel one of the signed-in employee's own leave applications, for example one HELD with unpaid "
                       "days. confirm=false shows what would be cancelled; confirm=true cancels it.",
        "services": ["lv_list", "lv_cancel"],
        "input_schema": {"type": "object", "properties": {
            "leave_no": {"type": "string", "description": "e.g. LV/2026/00388"},
            "reason": {"type": "string", "description": "why, in the user's words"},
            "confirm": {"type": "boolean", "description": "false (default) = preview; true = cancel"}},
            "required": ["leave_no", "reason"]},
        "output_schema": {"type": "object", "properties": {"ok": {"type": "boolean"}, "status": {"type": "string"}}},
        "example": {"leave_no": "LV/2026/00388", "reason": "applied with the wrong type"}}, indent=2))
    (d / "tool.py").write_text('''from bahi.agent.tools import ToolError, _bahi


def run(s, leave_no, reason, confirm=False):
    rows = _bahi(s, "lv_list", lv_no=leave_no)["rows"]
    if not rows:
        raise ToolError(f"No application {leave_no}.", "Check the number with leave_status.")
    app = rows[0]
    if not confirm:
        return {"ok": True, "preview": app, "note": "Nothing cancelled yet. Ask the user to confirm."}
    if app["status"] != "CN":
        _bahi(s, "lv_cancel", lv_no=leave_no, reason=reason)
    return {"ok": True, "leave_no": leave_no, "status": _bahi(s, "lv_list", lv_no=leave_no)["rows"][0]["status"]}
''')
print("lint:", lint_folder(FOLDER, "hr_leave") or "clean")

lint: clean


## 4 · Test the promises with code, before any model

Three tests, no model: the preview saves nothing; the silent outcome comes back as a warning; a lost reply leaves
exactly one application.

In [9]:
erp = ERP(modules=["hr_leave"])
tb = ModuleToolbox(erp, "KJOSHI", folder=FOLDER)
args = {"leave_type": "CL", "from_date": "2026-10-29", "to_date": "2026-10-30", "reason": "sister's wedding"}
pv = tb.call("apply_leave", args)
print("(a) preview:", pv["preview"], "| saved:", len(erp.q("SELECT * FROM lv_app WHERE from_dt='2026-10-29'")))
print("    warning:", pv["warnings"])
held = tb.call("apply_leave", {**args, "confirm": True})
print("(b) receipt:", held["status"], "|", held.get("warning"))

(a) preview: {'employee': 'Kavita Joshi (E-1042)', 'leave_type': 'CL', 'from_date': '2026-10-29', 'to_date': '2026-10-30', 'days': 2, 'balance': 1.0, 'unpaid_days': 1.0, 'approver': 'RMEHTA'} | saved: 0
(b) receipt: HELD: unpaid days, not sent to the approver | Saved but HELD: 1.0 day(s) without pay, and it has NOT gone to the approver. Tell the user plainly; cancel it and apply with a type that covers all the days.


### TODO (exercise 2): the lost-reply test
In a fresh `ERP(modules=["hr_leave"])`, inject `erp.inject("lv_save", "timeout_after_commit", 1)`, call
`apply_leave` with EL and `confirm=True` twice, and prove there is exactly one application for 29 Oct.

In [10]:
if SOLUTION:
    erp = ERP(modules=["hr_leave"])
    tb = ModuleToolbox(erp, "KJOSHI", folder=FOLDER)
    erp.inject("lv_save", "timeout_after_commit", 1)
    el = {**args, "leave_type": "EL", "confirm": True}
    r1, r2 = tb.call("apply_leave", el), tb.call("apply_leave", el)
    n = len(erp.q("SELECT * FROM lv_app WHERE from_dt='2026-10-29' AND status!='CN'"))
    print(r1.get("leave_no"), r1.get("status"), "|", r2.get("leave_no"), "already_done:", r2.get("already_done"),
          "| applications:", n)
    assert n == 1

LV/2026/00388 WAITING FOR APPROVAL | LV/2026/00388 already_done: True | applications: 1


## 5 · Served over MCP

The same folders, as an MCP server bound to Kavita's own session, with the purchasing toolbox's `search_policy`
beside them so Ask Rasoi can cite the leave policy.

In [11]:
from bahi.mcp_server import MCPClient, MCPClientTools

c = MCPClient(build_module_server(ERP(modules=["hr_leave"]), "KJOSHI", folder=FOLDER, with_purchasing=["search_policy"]))
print([t.name for t in c.list_tools()])
c.close()

['apply_leave', 'cancel_leave', 'check_leave_balance', 'leave_status', 'working_days', 'search_policy']


## 6 · Ask Rasoi on HR leave: the kit's four scenarios as the evaluation

The Lecture 5 loop, the Lecture 7 agent, a leave prompt (`LEAVE_PROMPT`), the tools over MCP. The kit's
`run_module_scenario` plays Kavita (or Neha), lets the approver approve, and checks Bahi's tables in code.
Our recorded run, run 1 of three (the other two are in `experiments/logs/leave_askrasoi.json`).
The replay uses the four folders `scaffold()` wrote (not your `cancel_leave`), exactly as recorded.

In [12]:
from bahi.modules._kit import load, run_module_scenario

for extra in ("cancel_leave",):  # keep the recorded tool set: the model's prompt lists every tool
    import shutil
    shutil.rmtree(FOLDER / extra, ignore_errors=True)
_, S, _ = load("hr_leave")
keep = []
res = [run_module_scenario(leave_agent(FLASH, "L8v3_leave", FOLDER, keep=keep), "hr_leave", sc) for sc in S.SCENARIOS]
pd.DataFrame([{"#": r.scenario, "scenario": r.name, "result": "pass" if r.passed else "FAIL",
               "never-events": "; ".join(r.never_events), "notes": "; ".join(r.notes)[:80], "₹": r.cost_inr,
               "last reply": (r.turns[-1]["agent"] or "")[:110]} for r in res])

,#,scenario,result,never-events,notes,₹,last reply
0,1,Two days off (the after sentence),pass,,,0.5508,Confirmed. Leave application LV/2026/00388 for 29 to 30 October 2026 (2 days EL) is now APPROVED...
1,2,Stores leave inside the Diwali run,FAIL,"NE4 LV/2026/00388 in the Diwali run approved by RMEHTA, not the plant head",,0.5534,"Leave application LV/2026/00388 (05 Nov 2026 to 06 Nov 2026, 2 days CL) is confirmed APPROVED by..."
2,3,"Saved this morning, but silently held",pass,,,0.1147,"No, Rohit has not approved it. \n\nYour application (LV/2026/00388) for Thursday 29 October and ..."
3,4,Comp-off against a worked holiday,FAIL,,no approved leave at the end,0.5931,Here is the preview of your leave application:\n\n- Leave type: CL (Casual Leave)\n- Dates: 2 No...


### TODO (exercise 3): read one run
Print the conversation of scenario 1 (`res[0].turns`) and the tool calls of its trace (`keep[0].trace`).
Which tool call decided whether a day would be unpaid?

In [13]:
if SOLUTION:
    for t in res[0].turns:
        print("KAVITA:", t["user"], "\nASK RASOI:", t["agent"], "\n")
    print([(st["tool"], st["args"]) for st in keep[0].trace if st["kind"] == "tool"])

KAVITA: I need Thursday and Friday off for my sister's wedding. 
ASK RASOI: Thursday 29 October and Friday 30 October count as 2 working days. Since you have 1 day of CL remaining, I have selected EL (balance: 9 days) so that both days are fully paid.

• Leave type: EL (Earned Leave)
• Dates: 29-10-2026 to 30-10-2026 (2 working days)
• Unpaid days: 0
• Approver: RMEHTA

Please confirm if you would like me to submit this application. 

KAVITA: Yes, go ahead. 
ASK RASOI: Your leave application (LV/2026/00388) has been submitted.

• Dates: 29-10-2026 to 30-10-2026 (2 days, EL)
• Unpaid days: 0
• Status: Waiting for approval by RMEHTA 

KAVITA: Rohit Mehta has approved LV/2026/00388. 
ASK RASOI: Confirmed. Leave application LV/2026/00388 for 29 to 30 October 2026 (2 days EL) is now APPROVED by RMEHTA. 

[('working_days', {'to_date': '2026-10-30', 'from_date': '2026-10-29'}), ('check_leave_balance', {}), ('apply_leave', {'to_date': '2026-10-30', 'from_date': '2026-10-29', 'reason': "Sister'

## 7 · Demo day

Your module's demo-day change is announced at the end of Lecture 11 (`modules/<kit>/CHANGE.md`). For HR leave it is
circular HR-CIR-2026-07: purchase staff now need the plant head for leave in the Diwali run. The hidden test asks
for 5 and 6 November off; it passes only if the leave is paid, goes to **Anand Patil**, and the reply names the rule.

### TODO (exercise 4): where would the change live?
Write three lines: which document you add to the knowledge, which tool argument `apply_leave` needs (Bahi's
`lv_save` has a `fwd_to` field), and which guard you add to the trust layer.

In [14]:
if SOLUTION:
    print("1. Knowledge: add hr-circular-2026-07.md (HR-CIR-2026-07) to the corpus; search_policy must find it.")
    print("2. Tool: apply_leave gets an optional forward_to, passed to lv_save as fwd_to (APATIL in the Diwali run).")
    print("3. Trust: a guard refusing leave for Purchase staff in 2-14 Nov unless forwarded to the plant head (NE4).")

1. Knowledge: add hr-circular-2026-07.md (HR-CIR-2026-07) to the corpus; search_policy must find it.
2. Tool: apply_leave gets an optional forward_to, passed to lv_save as fwd_to (APATIL in the Diwali run).
3. Trust: a guard refusing leave for Purchase staff in 2-14 Nov unless forwarded to the plant head (NE4).


## 8 · Your production stack

The rubric's line 7 (10 points): your module runs on at least two industry tools from Lectures 8–11, from two
different layers, each chosen with a written reason, and you measure one comparison yourself. The full menu is in
`lectures/L12-your-module/rubric.md`. Whatever runs the loop, writes still go through your tool folders.

### TODO (exercise 5): draft your STACK.md
Pick two layers for HR leave. For each: the tool and its pinned version, the hand-built piece it replaces or
watches, why this tool and not its rival, and the comparison you will measure on your own run.

In [15]:
STACK = {
    "observability": {"tool": "TODO", "replaces_or_watches": "TODO", "why": "TODO", "not_picked": "TODO", "compare": "TODO"},
    "evals": {"tool": "TODO", "replaces_or_watches": "TODO", "why": "TODO", "not_picked": "TODO", "compare": "TODO"},
}
if SOLUTION:
    STACK = {
        "observability": {"tool": "Langfuse (pin the version you install)", "replaces_or_watches": "the Lecture 7 trace, read by hand",
                          "why": "open source and self-hostable, so HR data stays on Rasoi's machines",
                          "not_picked": "LangSmith: hosted only for us, and no key in our build",
                          "compare": "the same failing scenario 2 run: the wrong step found in our trace vs in Langfuse"},
        "evals": {"tool": "DeepEval (pin the version you install)", "replaces_or_watches": "the Lecture 7 suite and gate",
                  "why": "runs the four scenarios as tests in CI, so a change cannot merge if a scenario breaks",
                  "not_picked": "promptfoo: kept for red-teaming, not as the gate",
                  "compare": "the 12 runs (4 scenarios x 3): our gate's verdict vs DeepEval's, case by case"},
    }
layers = [k for k, v in STACK.items() if v["tool"] != "TODO"]
print(f"{len(layers)} layers chosen:", ", ".join(layers) or "none yet")
for layer, v in STACK.items():
    print(f"\n## {layer}: {v['tool']}")
    for k in ("replaces_or_watches", "why", "not_picked", "compare"):
        print(f"- {k.replace('_', ' ')}: {v[k]}")

2 layers chosen: observability, evals

## observability: Langfuse (pin the version you install)
- replaces or watches: the Lecture 7 trace, read by hand
- why: open source and self-hostable, so HR data stays on Rasoi's machines
- not picked: LangSmith: hosted only for us, and no key in our build
- compare: the same failing scenario 2 run: the wrong step found in our trace vs in Langfuse

## evals: DeepEval (pin the version you install)
- replaces or watches: the Lecture 7 suite and gate
- why: runs the four scenarios as tests in CI, so a change cannot merge if a scenario breaks
- not picked: promptfoo: kept for red-teaming, not as the gate
- compare: the 12 runs (4 scenarios x 3): our gate's verdict vs DeepEval's, case by case
